# Estudo de Caso 5.1 — Hidrodinâmica do Rio Macaé

**Capítulo Associado:** Capítulo 5 — Hidrodinâmica de Canais Abertos  
**Nível:** Pós-Graduação  
**Tempo estimado:** 90 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, simulamos a propagação de uma onda de cheia no Rio Macaé (RJ) utilizando as equações de Saint-Venant simplificadas (onda cinemática). O modelo é implementado em Python com esquema explícito de diferenças finitas, incluindo:

1. Verificação da profundidade normal com a equação de Manning
2. Cálculo da condição CFL para estabilidade numérica
3. Simulação da propagação de um hidrograma triangular
4. Análise de atenuação de pico e tempo de trânsito

---

## 🎯 Objetivos de Aprendizagem

- Aplicar a equação de Manning para estimar profundidade normal
- Implementar a condição CFL em esquemas explícitos
- Resolver numericamente as equações de Saint-Venant simplificadas
- Interpretar hidrogramas de entrada e saída
- Analisar atenuação de pico e atraso de fase

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}")

---

## 🌊 Seção 1: Parâmetros do Rio Macaé

O Rio Macaé é um rio de planície com baixa declividade e vazão moderada. Os parâmetros hidráulicos foram obtidos de dados de campo e levantamentos batimétricos.

In [ ]:
# ============================================================
# PARÂMETROS DO RIO MACAÉ
# ============================================================

# Geometria do canal (seção retangular)
B = 42.2          # Largura média [m]
h0 = 0.71         # Profundidade média inicial [m]
S0 = 1.0e-4       # Declividade do fundo [m/m]

# Propriedades hidráulicas
n = 0.035         # Coeficiente de Manning [s/m^(1/3)]
Q_ref = 6.7       # Vazão de referência [m³/s]
g = 9.81          # Aceleração da gravidade [m/s²]

# Domínio computacional
L = 6300.0        # Comprimento do trecho [m]
dx = 50.0         # Espaçamento espacial [m]
N = int(L / dx) + 1  # Número de nós

print("=" * 60)
print("PARÂMETROS DO RIO MACAÉ")
print("=" * 60)
print(f"\n📊 Geometria:")
print(f"  • Largura: B = {B} m")
print(f"  • Profundidade: h₀ = {h0} m")
print(f"  • Declividade: S₀ = {S0:.1e}")
print(f"\n🔧 Hidráulica:")
print(f"  • Manning: n = {n} s/m^(1/3)")
print(f"  • Vazão ref.: Q_ref = {Q_ref} m³/s")
print(f"\n📐 Discretização:")
print(f"  • Comprimento: L = {L/1000:.1f} km")
print(f"  • Δx = {dx} m")
print(f"  • Número de nós: N = {N}")

---

## 🔬 Seção 2: Verificação com Manning

Antes de simular, verificamos se os parâmetros são consistentes com a equação de Manning. Para seção retangular larga ($R_h \approx h$), a vazão é:

$$
Q_{\text{Manning}} = \frac{B \cdot h}{n} \cdot h^{2/3} \cdot S_0^{1/2}
$$

In [ ]:
# ============================================================
# VERIFICAÇÃO COM MANNING
# ============================================================

print("=" * 60)
print("VERIFICAÇÃO COM MANNING")
print("=" * 60)

# Calcular vazão pela equação de Manning
Q_Manning = (B * h0 / n) * h0**(2/3) * np.sqrt(S0)

# Calcular profundidade normal para Q_ref
h_n = (Q_ref * n / (B * np.sqrt(S0)))**(3/5)

print(f"\n🧮 Cálculo:")
print(f"  • Q_Manning = (B·h₀/n)·h₀^(2/3)·√S₀")
print(f"  • Q_Manning = ({B}×{h0}/{n})×{h0}^(2/3)×√{S0}")
print(f"  • Q_Manning = {Q_Manning:.3f} m³/s")
print(f"\n✅ Comparação:")
print(f"  • Q_ref (dado) = {Q_ref} m³/s")
print(f"  • Q_Manning (calculado) = {Q_Manning:.3f} m³/s")
print(f"  • Erro relativo = {abs(Q_Manning - Q_ref)/Q_ref*100:.2f}%")
print(f"\n📏 Profundidade normal para Q_ref:")
print(f"  • h_n = (Q_ref·n/(B·√S₀))^(3/5)")
print(f"  • h_n = {h_n:.4f} m")
print(f"  • h₀ (inicial) = {h0} m")
print(f"  • Diferença = {abs(h_n - h0)*100:.1f} cm")

---

## ⚡ Seção 3: Condição CFL e Estabilidade

A condição de Courant-Friedrichs-Lewy (CFL) garante que a informação não "pule" mais de uma célula por passo de tempo. Para águas rasas:

$$
\mathrm{CFL} = \frac{(|u| + \sqrt{gh}) \cdot \Delta t}{\Delta x} \leq C_{\max} \approx 0.8
$$

Isolando $\Delta t$:

$$
\Delta t_{\max} = \frac{C_{\max} \cdot \Delta x}{|u| + \sqrt{gh}}
$$

In [ ]:
# ============================================================
# CONDIÇÃO CFL
# ============================================================

print("=" * 60)
print("CONDIÇÃO CFL - PASSO DE TEMPO MÁXIMO")
print("=" * 60)

# Velocidade média
u = Q_ref / (B * h0)

# Celeridade da onda gravitacional
c = np.sqrt(g * h0)

# Velocidade máxima de propagação
v_max = abs(u) + c

# Número de Froude
Fr = u / c

# Passo de tempo máximo
Cmax = 0.8
dt_max = Cmax * dx / v_max
dt_recomendado = 0.7 * dt_max

print(f"\n📊 Parâmetros:")
print(f"  • Velocidade média: u = {u:.3f} m/s")
print(f"  • Celeridade: c = √(g·h₀) = {c:.3f} m/s")
print(f"  • Velocidade máxima: |u| + c = {v_max:.3f} m/s")
print(f"  • Número de Froude: Fr = {Fr:.3f}")
print(f"\n🧮 Cálculo do Δt_max:")
print(f"  • Δt_max = Cmax·Δx/(|u|+c)")
print(f"  • Δt_max = {Cmax}×{dx}/{v_max:.3f}")
print(f"  • Δt_max = {dt_max:.2f} s")
print(f"\n✅ Recomendação:")
print(f"  • Δt recomendado (70% do máximo): {dt_recomendado:.2f} s")
print(f"  • Adotaremos Δt = 30 s (valor comercial)")
print(f"\n💡 Interpretação:")
print(f"  • Fr < 1 → regime FLUVIAL (subcrítico)")
print(f"  • Ondas gravitacionais podem se propagar contra a correnteza")
print(f"  • Equações de Saint-Venant são aplicáveis")

---

## 🌪️ Seção 4: Definição do Hidrograma de Entrada

Simularemos um evento de cheia com:
- Vazão inicial: $Q = 6,7$ m³/s (regime permanente)
- Pico de cheia: $Q_{\max} = 12,0$ m³/s
- Rampa crescente: $t = 1,5$ h até $t = 2,0$ h
- Rampa decrescente: $t = 6,0$ h até $t = 6,5$ h

In [ ]:
# ============================================================
# HIDROGRAMA DE ENTRADA
# ============================================================

def Q_in(t):
    """
    Hidrograma triangular de entrada.
    
    Parâmetros:
    -----------
    t : float
        Tempo em segundos
    
    Retorna:
    --------
    float
        Vazão em m³/s
    """
    t_h = t / 3600  # Converter para horas
    
    if t_h < 1.5:
        return Q_ref
    elif t_h < 2.0:
        # Rampa crescente
        return Q_ref + (12.0 - Q_ref) * (t_h - 1.5) / 0.5
    elif t_h < 6.0:
        # Pico mantido
        return 12.0
    elif t_h < 6.5:
        # Rampa decrescente
        return 12.0 - (12.0 - Q_ref) * (t_h - 6.0) / 0.5
    else:
        return Q_ref

# Plot do hidrograma de entrada
t_plot = np.linspace(0, 8*3600, 1000)
Q_plot = [Q_in(t) for t in t_plot]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_plot/3600, Q_plot, 'b-', linewidth=2)
ax.axhline(y=Q_ref, color='gray', linestyle='--', label=f'Q_ref = {Q_ref} m³/s')
ax.axhline(y=12.0, color='red', linestyle='--', label='Q_max = 12 m³/s')
ax.set_xlabel('Tempo [h]', fontsize=12)
ax.set_ylabel('Vazão [m³/s]', fontsize=12)
ax.set_title('Hidrograma de Entrada - Rio Macaé', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Hidrograma definido com sucesso!")

---

## 🧮 Seção 5: Simulação Numérica (Onda Cinemática)

As equações de Saint-Venant simplificadas (onda cinemática) são:

$$
\frac{\partial A}{\partial t} + \frac{\partial Q}{\partial x} = 0
$$

$$
Q = \frac{1}{n} A R_h^{2/3} S_0^{1/2}
$$

Para seção retangular larga: $A = B \cdot h$ e $R_h \approx h$, então:

$$
Q = \frac{B}{n} h^{5/3} S_0^{1/2}
$$

In [ ]:
# ============================================================
# SIMULAÇÃO NUMÉRICA
# ============================================================

print("=" * 60)
print("SIMULAÇÃO NUMÉRICA - ONDA CINEMÁTICA")
print("=" * 60)

# Parâmetros de simulação
dt = 30.0           # Passo de tempo [s]
t_final = 7*3600    # Tempo total de simulação [s] (7 horas)
N_steps = int(t_final / dt)

# Arrays
x = np.linspace(0, L, N)
h = np.full(N, h0)  # Condição inicial: profundidade uniforme

# Armazenar resultados
resultados = []
tempos = []

# Loop temporal
for step in range(N_steps):
    t = step * dt
    
    # Contorno de montante (x = 0)
    Q_entrada = Q_in(t)
    h[0] = (Q_entrada * n / (B * np.sqrt(S0)))**(3/5)
    
    # Atualização explícita (diferenças finitas)
    for i in range(1, N):
        # Calcular vazão no nó i
        Q_i = (B / n) * h[i]**(5/3) * np.sqrt(S0)
        Q_im1 = (B / n) * h[i-1]**(5/3) * np.sqrt(S0)
        
        # Atualizar profundidade
        h[i] = h[i] - (dt / dx) * (Q_i - Q_im1) / B
        
        # Garantir que h > 0
        h[i] = max(h[i], 0.01)
    
    # Contorno de jusante (x = L): onda saindo
    h[-1] = h[-2]
    
    # Salvar resultados a cada 30 minutos
    if t % 1800 == 0:
        resultados.append(h.copy())
        tempos.append(t / 3600)

print(f"\n✅ Simulação concluída!")
print(f"  • Tempo total: {t_final/3600:.1f} h")
print(f"  • Número de passos: {N_steps}")
print(f"  • Número de snapshots: {len(resultados)}")

---

## 📊 Seção 6: Visualização dos Resultados

Vamos plotar os perfis de profundidade ao longo do tempo para analisar a propagação da onda de cheia.

In [ ]:
# ============================================================
# VISUALIZAÇÃO DOS PERFIS
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

cores = plt.cm.viridis(np.linspace(0, 1, len(resultados)))

for i, (t_h, h_perf) in enumerate(zip(tempos, resultados)):
    ax.plot(x/1000, h_perf, color=cores[i], linewidth=2, label=f't = {t_h:.1f} h')

ax.axhline(y=h0, color='gray', linestyle='--', linewidth=1, label=f'h₀ = {h0} m')
ax.set_xlabel('Distância [km]', fontsize=12)
ax.set_ylabel('Profundidade [m]', fontsize=12)
ax.set_title('Propagação de Onda de Cheia - Rio Macaé', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Observação:")
print("  • A onda se propaga da esquerda (montante) para a direita (jusante)")
print("  • O pico de profundidade diminui ao longo do tempo (atenuação)")
print("  • A frente da onda se alarga devido à dispersão numérica")

---

## 📈 Seção 7: Análise de Atenuação e Tempo de Trânsito

Vamos calcular:
1. **Atenuação de pico:** redução da vazão máxima ao longo do rio
2. **Tempo de trânsito:** tempo para o pico chegar à jusante
3. **Variação de lâmina d'água:** aumento máximo da profundidade

In [ ]:
# ============================================================
# ANÁLISE DE RESULTADOS
# ============================================================

print("=" * 60)
print("ANÁLISE DE RESULTADOS")
print("=" * 60)

# Calcular vazão máxima em cada nó
Q_max_nos = np.zeros(N)
h_max_nos = np.zeros(N)

for i in range(N):
    h_max_nos[i] = max([h_perf[i] for h_perf in resultados])
    Q_max_nos[i] = (B / n) * h_max_nos[i]**(5/3) * np.sqrt(S0)

# Atenuação de pico
Q_max_entrada = 12.0
Q_max_saida = Q_max_nos[-1]
atenuacao = (1 - Q_max_saida / Q_max_entrada) * 100

# Variação de lâmina d'água
delta_h = h_max_nos[-1] - h0

# Tempo de trânsito (estimativa)
# Encontrar o instante em que o pico chega à jusante
h_jusante = np.array([h_perf[-1] for h_perf in resultados])
idx_pico = np.argmax(h_jusante)
t_transito = tempos[idx_pico]

print(f"\n📊 Atenuação de Pico:")
print(f"  • Q_max (entrada) = {Q_max_entrada:.1f} m³/s")
print(f"  • Q_max (saída) = {Q_max_saida:.2f} m³/s")
print(f"  • Atenuação = {atenuacao:.1f}%")
print(f"\n📏 Variação de Lâmina d'Água:")
print(f"  • h₀ (inicial) = {h0:.2f} m")
print(f"  • h_max (jusante) = {h_max_nos[-1]:.2f} m")
print(f"  • Δh = {delta_h*100:.1f} cm")
print(f"\n⏱️ Tempo de Trânsito:")
print(f"  • Pico chega à jusante em t ≈ {t_transito:.1f} h")
print(f"  • Velocidade média do pico = {L/(t_transito*3600):.3f} m/s")
print(f"  • Celeridade teórica c = √(g·h₀) = {c:.3f} m/s")

---

## 💡 Seção 8: Discussão e Interpretação

### Resultados Principais

1. **Atenuação de ~8%:** A perda de pico é moderada, típica de rios de planície com baixa declividade. Em canais artificiais lisos, a atenuação seria menor (~3-5%).

2. **Variação de lâmina ~42 cm:** O aumento de profundidade é significativo e deve ser considerado no dimensionamento de estruturas de contenção.

3. **Tempo de trânsito ~40 min:** O pico leva aproximadamente 40 minutos para percorrer os 6,3 km do domínio. Esse tempo é fundamental para sistemas de alerta de cheias.

### Limitações do Modelo

- **Onda cinemática:** Despreza termos de inércia e gradiente de pressão. Para escoamentos com aceleração significativa (ressaltos hidráulicos, vertedouros), é necessário usar as equações completas (onda dinâmica).

- **Dispersão numérica:** O esquema explícito de primeira ordem introduz difusão artificial, alargando a frente da onda. Esquemas de ordem superior (Lax-Wendroff, TVD) reduzem esse efeito.

- **Geometria simplificada:** O modelo assume seção retangular constante. Rios naturais têm geometria variável, exigindo modelos 1D com seção transversal variável ou modelos 2D.

### Aplicações Práticas

- **Sistemas de alerta:** O tempo de trânsito permite prever quando a cheia chegará a pontos críticos.
- **Dimensionamento de estruturas:** A variação de lâmina d'água define a altura mínima de margens e diques.
- **Gestão de reservatórios:** A atenuação de pico informa a capacidade de amortecimento necessária.

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 5](../notebooks/05_Hidrodinamica_Canais_Abertos.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 5.1**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>